In [1]:
import sys
import pandas as pd
import pm4py

df = pd.read_csv("../data/BPI Challenge 2017.csv")
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], format="mixed")
df = pm4py.format_dataframe(df, case_id="case:concept:name",
                            activity_key="concept:name", timestamp_key="time:timestamp")

# A_ and O_ events completely, W_ work items only by their "start" event
log = df[(df["EventOrigin"] != "Workflow") | (df["lifecycle:transition"] == "start")]
log = log.sort_values(["case:concept:name", "time:timestamp"], kind="stable")

# merge directly repeated starts of the same work item
prev = log.groupby("case:concept:name")["concept:name"].shift()
log = log[(log["concept:name"] != prev).fillna(True).astype(bool)]
print("Events:", len(log))



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




Events: 552624


In [2]:
n_var_all = df.groupby("case:concept:name")["concept:name"].agg(tuple).nunique()
n_var_log = log.groupby("case:concept:name")["concept:name"].agg(tuple).nunique()
print("Variants all events:", n_var_all, "| Variants model log:", n_var_log)

Variants all events: 15930 | Variants model log: 3815


In [3]:
def evaluate(log, net, im, fm, name):
    fit = pm4py.fitness_token_based_replay(log, net, im, fm)
    prec = pm4py.precision_token_based_replay(log, net, im, fm)
    gen = pm4py.generalization_tbr(log, net, im, fm)
    simp = pm4py.simplicity_petri_net(net, im, fm)
    return {"model": name, "transitions": len(net.transitions),
            "fitting cases %": round(fit["percentage_of_fitting_traces"], 1),
            "fitness": round(fit["log_fitness"], 3), "precision": round(prec, 3),
            "generalization": round(gen, 3), "simplicity": round(simp, 3)}

In [4]:
results, models = [], {}
for nt in [0.0, 0.05, 0.1, 0.15]:
    net, im, fm = pm4py.discover_petri_net_inductive(log, noise_threshold=nt)
    models[nt] = (net, im, fm)
    results.append(evaluate(log, net, im, fm, f"IM, noise {nt}"))
results_df = pd.DataFrame(results)
results_df.to_csv("../results/model_comparison.csv", index=False)
results_df

C:\Users\User\Documents\bpic17-process-mining\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
replaying log with TBR, completed traces :: 100%|██████████| 3815/3815 [00:08<00:00, 443.03it/s]


,model,transitions,fitting cases %,fitness,precision,generalization,simplicity
0,"IM, noise 0.0",66,100.0,1.000,0.269,0.950,0.636
1,"IM, noise 0.05",63,96.5,0.998,0.387,0.975,0.647
2,"IM, noise 0.1",56,82.6,0.990,0.441,0.971,0.649
3,"IM, noise 0.15",53,57.5,0.969,0.535,0.970,0.669


In [7]:
net1, im1, fm1 = models[0.1]
pm4py.save_vis_petri_net(net1, im1, fm1, "../figures/petri_net_final.png")
align_fit = pm4py.fitness_alignments(log, net1, im1, fm1)
align_prec = pm4py.precision_alignments(log, net1, im1, fm1)
print("Alignments – fitting cases %:", round(align_fit["percentage_of_fitting_traces"], 1))
print("Alignments – fitness:        ", round(align_fit["log_fitness"], 3))
print("Alignments – precision:      ", round(align_prec, 3))

aligning log, completed variants :: 100%|██████████| 3815/3815 [09:39<00:00,  6.58it/s]
computing precision with alignments, completed variants :: 100%|██████████| 21768/21768 [06:37<00:00, 54.76it/s] 


Alignments – fitting cases %: 82.0
Alignments – fitness:         0.988
Alignments – precision:       0.443


In [2]:
sys.path.append("../src")
import bpmn_layout as bl

tree = pm4py.discover_process_tree_inductive(log, noise_threshold=0.1)
print(tree)

conditions = {"A_Cancelled": "not returned",
              "O_Accepted": "offer returned",
              "A_Denied": "denied by bank"}
rules = ("Decision rules (decision tree, accuracy 0.85):\n"
         "not returned → Cancelled (98 %)\n"
         "returned, complete → Accepted (85 %), Denied (9 %)\n"
         "returned, incomplete → Accepted (67 %), Denied (32 %)")

d = bl.layout(tree, conditions, split_at=(4, 5))   # three rows
gw = [d.shapes[e["src"]] for e in d.edges
      if d.shapes.get(e["tgt"], {}).get("name") == "A_Cancelled"][0]
bl.add_annotation(d, "A_Cancelled", rules, dx=-20, dy=560 - gw["y"], w=440, h=80)

with open("../results/model_final.bpmn", "w", encoding="utf-8") as f:
    f.write(bl.to_bpmn_xml(d))
with open("../figures/bpmn_final.svg", "w", encoding="utf-8") as f:
    f.write(bl.to_svg(d))
print("saved results/model_final.bpmn and figures/bpmn_final.svg")

->( 'A_Create Application', X( tau, 'A_Submitted' ), X( 'W_Shortened completion ', ->( X( tau, 'W_Assess potential fraud' ), X( tau, 'W_Handle leads' ), +( *( 'W_Complete application', tau ), 'A_Concept' ) ) ), 'A_Accepted', *( ->( 'O_Create Offer', 'O_Created', X( tau, ->( 'O_Sent (mail and online)', X( tau, ->( 'W_Call after offers', 'A_Complete' ) ), X( tau, +( *( 'W_Validate application', tau ), X( tau, *( 'W_Call incomplete files', tau ) ), *( 'A_Validating', tau ), *( 'O_Returned', tau ), X( tau, *( 'A_Incomplete', tau ) ) ) ) ) ) ), tau ), X( tau, ->( X( 'A_Cancelled', ->( 'O_Accepted', 'A_Pending' ) ), X( tau, *( 'O_Cancelled', tau ) ) ) ), X( tau, 'W_Personal Loan collection' ), X( tau, ->( 'A_Denied', 'O_Refused' ) ) )
saved results/model_final.bpmn and figures/bpmn_final.svg


In [9]:
%pip install svglib

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 382.5 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 382.5 kB/s eta 0:00:04
   --------------- ------------------------ 0.8/2.0 MB 423.8 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 423.8 kB/s eta 0:00:03
   -------------------- ------------------- 1.0/2.0 MB 484.5 kB/s eta 0:00:03
   -------------------- ------------------- 1.0/2.0 MB 484.5 kB/s eta 0:00:03
   -------------------- ------------------- 1.0/2


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
bl.svg_to_pdf("../figures/bpmn_final.svg", "../figures/bpmn_final.pdf")
print("saved figures/bpmn_final.pdf")

saved figures/bpmn_final.pdf
